In [0]:

import os, sys
try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    if "/repoint/" in notebook_path:
        repo_root = "/Workspace" + notebook_path.rsplit("/repoint/", 1)[0]
    else:
        repo_root = "/Workspace/Users/extkgentekaki@cepal.gr/gr.cepal.dna.de.delivery"
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname
from helpers import delete_from_unity_table

env = get_env_prefix()
env_name = get_env_fullname()

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    # run_date empty: fall back to max business date
    run_date_rows = spark.sql(f'''
        SELECT MAX(BusinessDate) AS BusinessDate FROM {env_name}.derivative.dexposure
        UNION
        SELECT MAX(BusinessDate) AS BusinessDate FROM {env_name}.landed.profits_loan_details
    ''').collect()
    if len(run_date_rows) != 1:
        raise ValueError(
            f"run_date mismatch: dexposure and profits_loan_details have different MAX(BusinessDate). "
            f"Got {[str(r['BusinessDate']) for r in run_date_rows]}"
        )
    run_date = str(run_date_rows[0]['BusinessDate'])

# exprt_month = YYYYMM of the PREVIOUS month's last WORKING day, from ref_calendar
# (holiday/weekend aware, year-boundary safe). CalendarDate is a string column.
_cal = spark.sql(f'''
    SELECT PreviousMonthLastWorkingDate AS LastWorkingDayPrevMonth
    FROM {env_name}.reference.ref_calendar
    WHERE CalendarDate = '{run_date}'
''').collect()
if not _cal or _cal[0]['LastWorkingDayPrevMonth'] is None:
    raise ValueError(f"No PreviousMonthLastWorkingDate found in ref_calendar for run_date={run_date}")
exprt_month = str(_cal[0]['LastWorkingDayPrevMonth'])[:7].replace('-', '')   # e.g. '202511'

target_area = f"`{env}-cdh-delivery`.repoint"
partition_columns = ["BusinessDate"]

print(f"run_date: {run_date} | exprt_month: {exprt_month} | target: {target_area}")

In [0]:
from pyspark.sql.functions import (
    col, lit, when, coalesce, substring, date_format, to_date, try_to_date, broadcast,
    max as spark_max, current_timestamp, row_number
)
from pyspark.sql.types import DecimalType, StringType, DateType
from pyspark.sql.window import Window

run_month = f"{run_date[:4]}{run_date[5:7]}"

dates_row = spark.sql(f"""
    SELECT
        MAX(CASE WHEN CASE_MONTH = '{exprt_month}' THEN BusinessDate END) AS latest_bd_export,
        MAX(CASE WHEN CASE_MONTH = '{run_month}' AND BusinessDate <= DATE('{run_date}') THEN BusinessDate END) AS latest_bd_run
    FROM {env_name}.landed.qca_fs_cases_snap
    WHERE CASE_MONTH IN ('{exprt_month}', '{run_month}')
""").first()

latest_bd_export = dates_row["latest_bd_export"]
latest_bd_run = dates_row["latest_bd_run"]

if latest_bd_export is None:
    raise ValueError(f"No data for CASE_MONTH={exprt_month}. Check run_date={run_date}.")
if latest_bd_run is None:
    # Fallback: if no run_month data yet, use export month only
    latest_bd_run = latest_bd_export
    run_month = exprt_month

latest_bd = latest_bd_run
print(f"run_month: {run_month} | latest_bd_run: {latest_bd_run} | latest_bd_export: {latest_bd_export} | exprt_month: {exprt_month}")

def _load(table, extra_filter=None):
    df = spark.table(table).filter(col("BusinessDate") == latest_bd)
    return df.filter(extra_filter) if extra_filter is not None else df

_cl_filter = (
    (col("CASE_BU") == "074-CL") &
    (col("CASE_RELATION_TYPE_CODE") != 5000)
)
_src = spark.table(f"{env_name}.landed.qca_fs_cases_snap")

export_snap = _src.filter(
    (col("BusinessDate") == latest_bd_export) & (col("CASE_MONTH") == exprt_month) & _cl_filter
)
export_codes = export_snap.select("CASE_CODE").distinct()

fresh_snap = _src.filter(
    (col("BusinessDate") == latest_bd_run) & (col("CASE_MONTH") == run_month) & _cl_filter
)
fresh_codes = fresh_snap.select("CASE_CODE").distinct()

export_only_codes = export_codes.join(fresh_codes, "CASE_CODE", "left_anti")

fs_cases_snap_raw = (
    fresh_snap.join(export_codes, "CASE_CODE", "inner")
    .unionByName(
        export_snap.join(export_only_codes, "CASE_CODE", "inner")
    )
)

export_dpd_bucket = export_snap.select(
    col("CASE_CODE").alias("EDB_CASE_CODE"),
    col("CASE_DELINQUENT_DAYS").alias("EXPORT_DPD"),
    col("CASE_BUCKET").alias("EXPORT_BUCKET"),
)
fs_cases_snap = (
    fs_cases_snap_raw
    .join(export_dpd_bucket, col("CASE_CODE") == col("EDB_CASE_CODE"), "left")
    .withColumn("CASE_DELINQUENT_DAYS", coalesce(col("EXPORT_DPD"), col("CASE_DELINQUENT_DAYS")))
    .withColumn("CASE_BUCKET", coalesce(col("EXPORT_BUCKET"), col("CASE_BUCKET")))
    .drop("EDB_CASE_CODE", "EXPORT_DPD", "EXPORT_BUCKET")
)
dim_customer_static      = _load(f"{env_name}.landed.qca_dim_customer_static")
fs_customers_snap        = _load(f"{env_name}.landed.qca_fs_customers_snap", col("cust_slc_id") == "ALL")
fs_customers_snap_extra  = _load(f"{env_name}.landed.qca_fs_customers_snap_extra")
dim_cases_static         = _load(f"{env_name}.landed.qca_dim_cases_static")
fs_cases_snap_extra      = _load(f"{env_name}.landed.qca_fs_cases_snap_extra")
dim_bucket               = _load(f"{env_name}.landed.qca_dim_bucket")
dim_subproduct           = _load(f"{env_name}.landed.qca_dim_subproduct")
dim_subproduct_hier1     = _load(f"{env_name}.landed.qca_dim_subproduct_hier1")
dim_queue                = _load(f"{env_name}.landed.qca_dim_queue")
dim_packet               = _load(f"{env_name}.landed.qca_dim_packet")
dim_packet_hier3         = _load(f"{env_name}.landed.qca_dim_packet_hier3")
dim_external_company     = _load(f"{env_name}.landed.qca_dim_external_company")
dim_customer_category    = _load(f"{env_name}.landed.qca_dim_customer_category")
dim_legal_process_status = _load(f"{env_name}.landed.qca_dim_legal_process_status")
fsad_debts               = _load(f"{env_name}.landed.qca_fsad_debts")
fsad_customers_generic_open = _load(f"{env_name}.landed.qca_fsad_customers_generic_open")
fsad_list_of_values      = _load(f"{env_name}.landed.qca_fsad_list_of_values")
fs_arrangements          = _load(f"{env_name}.landed.qca_fs_arrangements")
dim_settlement_type      = _load(f"{env_name}.landed.qca_dim_settlement_type")
dim_settlement_status    = _load(f"{env_name}.landed.qca_dim_settlement_status")
dim_branch               = _load(f"{env_name}.landed.qca_dim_branch")

count = fs_cases_snap.count()
print(f"Loaded {count:,} fs_cases_snap rows")

In [0]:
src = spark.table(f"{env_name}.landed.qca_fs_cases_snap")

month_max_bd = (
    src.filter(col("CASE_MONTH") < run_month)
    .groupBy("CASE_MONTH")
    .agg(spark_max("BusinessDate").alias("max_bd"))
)

prior_cases = (
    src.alias("s")
    .join(month_max_bd.alias("m"),
          (col("s.CASE_MONTH") == col("m.CASE_MONTH")) &
          (col("s.BusinessDate") == col("m.max_bd")))
    .filter(
        (col("s.CASE_BU") == "074-CL") &
        (col("s.CASE_RELATION_TYPE_CODE") != 5000)
    )
    .select(
        col("s.CASE_CODE"),
        col("s.CASE_MONTH"),
        col("s.PACKH3_ID"),
        col("s.EXTC_CODE")
    )
)

w = Window.partitionBy("CASE_CODE").orderBy(col("CASE_MONTH").desc())

prev_packh3 = (
    prior_cases.filter(col("PACKH3_ID") != -100)
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .select(
        col("CASE_CODE").alias("PREV_CASE_CODE_P"),
        col("PACKH3_ID").alias("PREV_PACKH3_ID")
    )
)

prev_extc = (
    prior_cases.filter(col("EXTC_CODE") != -100)
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .select(
        col("CASE_CODE").alias("PREV_CASE_CODE_E"),
        col("EXTC_CODE").alias("PREV_EXTC_CODE")
    )
)

prev_month_snap = (
    prev_packh3.join(prev_extc,
                     col("PREV_CASE_CODE_P") == col("PREV_CASE_CODE_E"), "full")
    .select(
        coalesce(col("PREV_CASE_CODE_P"), col("PREV_CASE_CODE_E")).alias("PREV_CASE_CODE"),
        col("PREV_PACKH3_ID"),
        col("PREV_EXTC_CODE")
    )
)

_portfolio_codes = fs_cases_snap.select(col("CASE_CODE").alias("_PC")).distinct()

assign_src = (
    spark.table(f"{env_name}.primary.assignment")
    .filter(
        (col("SourceSystem") == "qca") &
        (col("BusinessDate") < "2024-01-30")   # only the gap period
    )
    .join(_portfolio_codes, col("CaseCode") == col("_PC"), "inner")
    .drop("_PC")
)

w_assign = Window.partitionBy("CaseCode").orderBy(col("BusinessDate").desc())

assign_prev_extc = (
    assign_src
    .filter(col("ExternalCompanyFullName").isNotNull())
    .withColumn("rn", row_number().over(w_assign))
    .filter(col("rn") == 1)
    .select(
        col("CaseCode").alias("A_CASE_E"),
        col("ExternalCompanyFullName").alias("A_PREV_EXTC_NAME")
    )
)

pkt_to_proc = (
    spark.table(f"{env_name}.primary.assignment")
    .filter(
        (col("SourceSystem") == "qca") &
        (col("BusinessDate") >= "2024-01-30") &
        col("PacketCode").isNotNull() &
        col("ProcessType").isNotNull() &
        (col("ProcessType") != "INTERNAL PRODUCTION")
    )
    .select("PacketCode", "ProcessType")
    .distinct()
)

assign_prev_pack = (
    assign_src
    .filter(col("PacketCode").isNotNull())
    .join(pkt_to_proc.alias("pm"), "PacketCode", "inner")  # keeps only non-internal packets
    .withColumn("rn", row_number().over(w_assign))
    .filter(col("rn") == 1)
    .select(
        col("CaseCode").alias("A_CASE_P"),
        col("pm.ProcessType").alias("A_PREV_PACKH3_DESC")
    )
)

assign_prev = (
    assign_prev_pack.join(assign_prev_extc,
        col("A_CASE_P") == col("A_CASE_E"), "full")
    .select(
        coalesce(col("A_CASE_P"), col("A_CASE_E")).alias("A_PREV_CASE_CODE"),
        col("A_PREV_PACKH3_DESC"),
        col("A_PREV_EXTC_NAME")
    )
)
_schema = assign_prev.schema
_rows = assign_prev.collect()
assign_prev = spark.createDataFrame(_rows, _schema)
print(f"assign_prev rows: {len(_rows):,}")

In [0]:
from pyspark.sql.functions import regexp_replace, broadcast

fs_cases_snap = fs_cases_snap.select(
    "CASE_CODE", "CUST_CODE", "CASE_BUCKET", "SPRD_CODE", "QUE_CODE",
    "PACK_CODE", "PACKH3_ID", "EXTC_CODE", "DEBT_LEGAL_PROCESS_STATUS_CODE",
    "DEBT_CODE", "SETE_CODE", "CASE_S_ID", "CASE_CONTR_NUM",
    "CAEX_CUSTOM_TEXT_30", "CASE_CURRENCY", "CASE_DELINQUENT_DAYS",
    "CASE_CUSTOM_DECIMAL_01", "CASE_OUTST_BALANCE", "MULTIPLY_FOR_LCY",
    "CASE_DEBT_AMOUNT", "CASE_MIN_PAYMENT", "CASE_FROZ_INT_AMT",
    "CASE_PROV_AMT", "CASE_WR_OF_AMOUNT", "CASE_DENOUNCE_STATUS",
    "CASE_DENOUNCE_DATE_VAR", "CASE_WR_OF_DATE", "CASE_LAST_PMT_AMT",
    "CASE_LAST_PMT_DATE", "CAEX_CUSTOM_DECIMAL_07", "CASE_OBJECTIVE_STATUS",
    "CASE_BU", "CASE_INTERNAL_BU", "CASE_EXP_DATE", "CASE_BILL_DATE",
    "CASE_DUE_DATE", "CASE_CREDIT_LIM", "CASE_LEGAL_PROCESS_STATUS_UPD",
    "SPET_CODE", "CAEX_CUSTOM_DATETIME_12", "CAEX_CUSTOM_DATETIME_11",
    "CAEX_CUSTOM_TEXT_15", "CAEX_CUSTOM_TEXT_44",
    "CSTM_DECIMAL_06", "CSTM_DECIMAL_07", "CSTM_DECIMAL_04", "CSTM_DECIMAL_05",
    "CASE_RELATION_TYPE_CODE",
)
dim_customer_static = dim_customer_static.select(
    "CUST_ID", "CUST_CIF", "CUST_AFM", "CUST_FIRST_NAME", "CUST_LAST_NAME",
    "CUST_FATH_NAME", "CUST_BIRTHDATE", "CUST_ID_TYPE", "CUST_ID_NUM",
    "CUST_GENERIC_INDICATOR_04",
)
fs_customers_snap = fs_customers_snap.select(
    "CUST_CODE", "CUST_GENERIC_TEXT_01", "CUST_CATEGORY",
    "CUST_LEGAL_PROCESS_STATUS", "CUCO_DEDICATED_PROPERTY_03_INTCODE",
    "CUCO_DEDICATED_PROPERTY_08_INTCODE", "CUEX_GENERIC_BIT_03",
    "CUEX_GENERIC_TEXT_03",
)
fs_customers_snap_extra = fs_customers_snap_extra.select(
    "CUST_CODE", "CUSX_GENERIC_BIT_02", "CUSX_GENERIC_NVARCHAR_01",
)
dim_cases_static = dim_cases_static.select(
    "CUST_CODE", "CASE_CODE", "CASE_S_ID", "CASE_CONTRACT_ACCOUNT_KEY",
    "CASE_LEGAL_CONTRACT", "CASE_APPLIC_NUM", "CASE_ACCOUNT_OPEN_DATE",
    "CAEX_CUSTOM_TEXT_27", "CASE_SOLD_FLAG", "CSTM_DATETIME_AD_03",
    "CSTM_TEXT_AD_20", "CAEX_CUSTOM_TEXT_16", "CASE_BRANCH_INTCODE",
    "CASE_ACCOUNT_FOR_SEARCH",
)
fs_cases_snap_extra = fs_cases_snap_extra.select(
    "CASE_CODE", "CEXT_GENERIC_BIT_02", "CEXT_GENERIC_NVARCHAR_01",
)
dim_bucket = dim_bucket.select("BCK_ID", "BCK_S_ID")
dim_subproduct = dim_subproduct.select("SPRD_ID", "SPRDH1_ID", "SPRD_S_ID", "SPRD_DESC")
dim_subproduct_hier1 = dim_subproduct_hier1.select("SPRDH1_ID", "SPRDH1_S_ID", "SPRDH1_DESC")
dim_queue = dim_queue.select("QUE_ID", "QUE_DESC", "QUEH1_ID")
dim_packet = dim_packet.select("PACK_ID", "PACK_ASSIGN_DATE")
dim_packet_hier3 = dim_packet_hier3.select("PACKH3_ID", "PACKH3_DESC")
dim_external_company = dim_external_company.select("EXTC_ID", "EXTC_NAME")
dim_customer_category = dim_customer_category.select("CUCAT_ID", "CUCAT_DESC")
fsad_debts = fsad_debts.select("DEBT_CODE", "DEBT_GENERIC_STATUS_03_DATE")
fsad_customers_generic_open = fsad_customers_generic_open.select(
    "CUST_CODE", "CUSG_GENERIC_DATETIME_07", "CUSG_GENERIC_DATETIME_05",
)
fs_arrangements = fs_arrangements.select("SETE_CODE", "SETY_ID", "SEST_ID")
dim_settlement_type = dim_settlement_type.select("SETY_ID", "SETY_DESC")
dim_settlement_status = dim_settlement_status.select("SEST_ID", "SEST_DESC")
dim_branch = dim_branch.select("BRN_INTCODE", "BRN_DESC")

# LOV sub-tables (already lean)
lov_interbank = fsad_list_of_values.filter(col("LOVT_CODE") == 834).select(
    col("LOV_CODE").alias("L1_CODE"), col("LOV_DESC").alias("L1_DESC"))
lov_customer_status = fsad_list_of_values.filter(col("LOVT_CODE") == 839).select(
    col("LOV_CODE").alias("L2_CODE"), col("LOV_DESC").alias("L2_DESC"))
lov_customer_l3869 = fsad_list_of_values.filter(col("LOVT_CODE") == 863).select(
    col("LOV_DESC").alias("L3_DESC"), col("LOV_COMMENTS").alias("L3_COMMENTS"))

lov_guarantee = (
    fsad_list_of_values.groupBy("LOV_DESC").agg(spark_max("LOV_CODE").alias("LOV_CODE"))
    .join(fsad_list_of_values.select(col("LOV_CODE"), col("LOV_COMMENTS").alias("LVAL_COMMENTS")), "LOV_CODE")
    .select(col("LOV_DESC").alias("LVAL_DESC"), "LVAL_COMMENTS")
)

lps_customer = dim_legal_process_status.select(col("LEPR_ID").alias("LPS2_ID"), col("LEPR_DESC").alias("LPS2_DESC"))
lps_debt = dim_legal_process_status.select(col("LEPR_ID").alias("LPS3_ID"), col("LEPR_DESC").alias("LPS3_DESC"))

prev_packet_hier3 = dim_packet_hier3.select(
    col("PACKH3_ID").alias("PREV_PACKH3_ID_DIM"),
    col("PACKH3_DESC").alias("PREV_PACKH3_DESC")
)
prev_external_company = dim_external_company.select(
    col("EXTC_ID").alias("PREV_EXTC_ID"),
    col("EXTC_NAME").alias("PREV_EXTC_NAME")
)

result = (
    fs_cases_snap
    .join(dim_customer_static, dim_customer_static["CUST_ID"] == fs_cases_snap["CUST_CODE"], "inner")
    .join(fs_customers_snap, fs_customers_snap["CUST_CODE"] == dim_customer_static["CUST_ID"], "left")
    .join(fs_customers_snap_extra, fs_customers_snap_extra["CUST_CODE"] == fs_cases_snap["CUST_CODE"], "left")
    .join(dim_cases_static, (dim_cases_static["CUST_CODE"] == dim_customer_static["CUST_ID"]) & (dim_cases_static["CASE_CODE"] == fs_cases_snap["CASE_CODE"]), "left")
    .join(fs_cases_snap_extra, fs_cases_snap_extra["CASE_CODE"] == fs_cases_snap["CASE_CODE"], "left")
    .join(broadcast(dim_bucket), dim_bucket["BCK_ID"] == fs_cases_snap["CASE_BUCKET"], "inner")
    .join(broadcast(dim_subproduct), dim_subproduct["SPRD_ID"] == fs_cases_snap["SPRD_CODE"], "inner")
    .join(broadcast(dim_subproduct_hier1), dim_subproduct_hier1["SPRDH1_ID"] == dim_subproduct["SPRDH1_ID"], "inner")
    .join(broadcast(dim_queue), dim_queue["QUE_ID"] == fs_cases_snap["QUE_CODE"], "inner")
    .join(dim_packet, dim_packet["PACK_ID"] == fs_cases_snap["PACK_CODE"], "inner")
    .join(broadcast(dim_packet_hier3), dim_packet_hier3["PACKH3_ID"] == fs_cases_snap["PACKH3_ID"], "inner")
    .join(broadcast(dim_external_company), dim_external_company["EXTC_ID"] == fs_cases_snap["EXTC_CODE"], "inner")
    .join(broadcast(dim_customer_category), dim_customer_category["CUCAT_ID"] == fs_customers_snap["CUST_CATEGORY"], "left")
    .join(broadcast(lps_customer), col("LPS2_ID") == fs_customers_snap["CUST_LEGAL_PROCESS_STATUS"], "left")
    .join(broadcast(lps_debt), col("LPS3_ID") == fs_cases_snap["DEBT_LEGAL_PROCESS_STATUS_CODE"], "left")
    .join(fsad_debts, fsad_debts["DEBT_CODE"] == fs_cases_snap["DEBT_CODE"], "left")
    .join(fsad_customers_generic_open, fsad_customers_generic_open["CUST_CODE"] == dim_customer_static["CUST_ID"], "left")
    .join(broadcast(lov_interbank), col("L1_CODE") == fs_customers_snap["CUCO_DEDICATED_PROPERTY_03_INTCODE"], "left")
    .join(broadcast(lov_customer_status), col("L2_CODE") == fs_customers_snap["CUCO_DEDICATED_PROPERTY_08_INTCODE"], "left")
    .join(broadcast(lov_customer_l3869), col("L3_DESC") == fs_customers_snap_extra["CUSX_GENERIC_NVARCHAR_01"], "left")
    .join(broadcast(lov_guarantee), col("LVAL_DESC") == fs_cases_snap_extra["CEXT_GENERIC_NVARCHAR_01"], "left")
    .join(fs_arrangements, fs_arrangements["SETE_CODE"] == fs_cases_snap["SETE_CODE"], "left")
    .join(broadcast(dim_settlement_type), dim_settlement_type["SETY_ID"] == fs_arrangements["SETY_ID"], "left")
    .join(broadcast(dim_settlement_status), dim_settlement_status["SEST_ID"] == fs_arrangements["SEST_ID"], "left")
    .join(broadcast(dim_branch), dim_branch["BRN_INTCODE"] == dim_cases_static["CASE_BRANCH_INTCODE"], "left")
    .join(prev_month_snap, col("PREV_CASE_CODE") == fs_cases_snap["CASE_CODE"], "left")
    .join(broadcast(prev_packet_hier3), prev_packet_hier3["PREV_PACKH3_ID_DIM"] == prev_month_snap["PREV_PACKH3_ID"], "left")
    .join(broadcast(prev_external_company), prev_external_company["PREV_EXTC_ID"] == prev_month_snap["PREV_EXTC_CODE"], "left")
    .join(assign_prev, col("A_PREV_CASE_CODE") == fs_cases_snap["CASE_CODE"], "left")
)

# Select final columns
final_df = result.select(
    lit(run_date).cast(DateType()).alias("BusinessDate"),
    lit(exprt_month).cast(StringType()).alias("EXPRT_MONTH"),
    fs_cases_snap["CASE_CODE"].cast(StringType()).alias("CASE_CODE"),
    dim_customer_static["CUST_CIF"].alias("CIF"),
    dim_customer_static["CUST_AFM"].alias("AFM"),
    dim_customer_static["CUST_FIRST_NAME"],
    dim_customer_static["CUST_LAST_NAME"],
    dim_customer_static["CUST_FATH_NAME"],
    fs_customers_snap["CUST_GENERIC_TEXT_01"].alias("MOTHERS_NAME"),
    date_format(dim_customer_static["CUST_BIRTHDATE"], "yyyy-MM-dd").alias("CUST_BIRTHDATE"),
    when(fs_customers_snap_extra["CUSX_GENERIC_BIT_02"].cast("string").isin("1", "255", "true", "True"), lit("Yes"))
        .otherwise(lit("No")).alias("CUST_DECEASED"),
    dim_customer_static["CUST_ID_TYPE"],
    dim_customer_static["CUST_ID_NUM"],
    dim_customer_static["CUST_GENERIC_INDICATOR_04"].alias("STAFF_INDICATOR"),
    fs_cases_snap["CASE_CONTR_NUM"].alias("ACCOUNT_NUMBER"),
    dim_cases_static["CASE_ACCOUNT_FOR_SEARCH"].alias("CLEAN_ACCOUNT_NUMBER"),
    when(fs_cases_snap["CASE_RELATION_TYPE_CODE"] == 5000, lit("Primary Owner"))
        .when(fs_cases_snap["CASE_RELATION_TYPE_CODE"] == 5001, lit("Co-Owner"))
        .when(fs_cases_snap["CASE_RELATION_TYPE_CODE"] == 5002, lit("Guarantor"))
        .otherwise(lit("NO_CATEGORY")).alias("RELATION_TYPE"),
    fs_cases_snap["CAEX_CUSTOM_TEXT_30"].alias("MASTER_ACCOUNT"),
    fs_cases_snap["CASE_CURRENCY"],
    when(dim_cases_static["CASE_S_ID"].like("074C%"), dim_cases_static["CASE_CONTRACT_ACCOUNT_KEY"])
        .otherwise(dim_cases_static["CASE_LEGAL_CONTRACT"]).alias("CONTRACT_NO"),
    dim_cases_static["CASE_APPLIC_NUM"].alias("APL_REF"),
    dim_bucket["BCK_S_ID"].alias("CURRENT_BUCKET"),
    date_format(dim_cases_static["CASE_ACCOUNT_OPEN_DATE"], "yyyy-MM-dd").alias("CASE_OPEN_DATE"),
    date_format(fs_cases_snap["CASE_EXP_DATE"], "yyyy-MM-dd").alias("CASE_EXP_DATE"),
    date_format(fs_cases_snap["CASE_BILL_DATE"], "yyyy-MM-dd").alias("BILLING_DATE"),
    date_format(fs_cases_snap["CASE_DUE_DATE"], "yyyy-MM-dd").alias("DUE_DATE"),
    fs_cases_snap["CASE_DELINQUENT_DAYS"].cast(StringType()).alias("DPD"),
    fs_cases_snap["CASE_CUSTOM_DECIMAL_01"].cast(StringType()).alias("CURRENT_DOSI"),
    (fs_cases_snap["CASE_OUTST_BALANCE"] * fs_cases_snap["MULTIPLY_FOR_LCY"]).cast(DecimalType(19, 2)).alias("OUTSTANDING_BALANCE"),
    (fs_cases_snap["CASE_DEBT_AMOUNT"] * fs_cases_snap["MULTIPLY_FOR_LCY"]).cast(DecimalType(19, 2)).alias("TOTAL_DUE"),
    (fs_cases_snap["CASE_MIN_PAYMENT"] * fs_cases_snap["MULTIPLY_FOR_LCY"]).cast(DecimalType(19, 2)).alias("MIN_PAY_DUE"),
    (fs_cases_snap["CASE_FROZ_INT_AMT"] * fs_cases_snap["MULTIPLY_FOR_LCY"]).cast(DecimalType(19, 2)).alias("FROZEN"),
    (fs_cases_snap["CASE_PROV_AMT"] * fs_cases_snap["MULTIPLY_FOR_LCY"]).cast(DecimalType(19, 2)).alias("MLT_AMOUNT"),
    (fs_cases_snap["CASE_WR_OF_AMOUNT"] * fs_cases_snap["MULTIPLY_FOR_LCY"]).cast(DecimalType(19, 2)).alias("WOFF_AMOUNT"),
    when(
        fs_cases_snap["CASE_DENOUNCE_STATUS"].like("%L") | fs_cases_snap["CASE_DENOUNCE_STATUS"].like("%WRITE%"),
        when(fs_cases_snap["CASE_DENOUNCE_DATE_VAR"].rlike(r"^\d{8}$"), date_format(try_to_date(fs_cases_snap["CASE_DENOUNCE_DATE_VAR"], "yyyyMMdd"), "yyyy-MM-dd")).otherwise(fs_cases_snap["CASE_DENOUNCE_DATE_VAR"])
    ).otherwise(date_format(fs_cases_snap["CASE_WR_OF_DATE"], "yyyy-MM-dd")).alias("WOFF_DATE"),
    (fs_cases_snap["CASE_LAST_PMT_AMT"] * fs_cases_snap["MULTIPLY_FOR_LCY"]).cast(DecimalType(19, 2)).alias("LAST_PAY_AMOUNT"),
    date_format(fs_cases_snap["CASE_LAST_PMT_DATE"], "yyyy-MM-dd").alias("LAST_PAY_DATE"),
    fs_cases_snap["CASE_DENOUNCE_STATUS"].alias("ACCOUNT_STATUS"),
    when(fs_cases_snap["CASE_DENOUNCE_DATE_VAR"].rlike(r"^\d{8}$"), date_format(try_to_date(fs_cases_snap["CASE_DENOUNCE_DATE_VAR"], "yyyyMMdd"), "yyyy-MM-dd")).otherwise(fs_cases_snap["CASE_DENOUNCE_DATE_VAR"]).alias("ACCOUNT_STATUS_DATE"),
    fs_cases_snap["CAEX_CUSTOM_DECIMAL_07"].cast(StringType()).alias("INTEREST_RATE"),
    fs_cases_snap["CASE_OBJECTIVE_STATUS"].alias("OBJECTIVE_STATUS"),
    date_format(fsad_debts["DEBT_GENERIC_STATUS_03_DATE"], "yyyy-MM-dd").alias("OBJECTIVE_STATUS_DATE"),
    dim_external_company["EXTC_NAME"].alias("AGENCY_CODE"),
    date_format(dim_packet["PACK_ASSIGN_DATE"], "yyyy-MM-dd").alias("ASSIGN_DATE"),
    dim_packet_hier3["PACKH3_DESC"].alias("PACKET_PROCESS_TYPE"),
    dim_queue["QUE_DESC"],
    when(dim_queue["QUEH1_ID"] == 261, lit("Predefined"))
        .when(dim_queue["QUEH1_ID"] == 263, lit("Manual"))
        .when(dim_queue["QUEH1_ID"] == 264, lit("Default"))
        .otherwise(dim_queue["QUEH1_ID"].cast(StringType())).alias("QUEUE_TYPE"),
    fs_cases_snap["CASE_BU"].alias("BU"),
    fs_cases_snap["CASE_INTERNAL_BU"].alias("HOST_BU"),
    substring(fs_cases_snap["CASE_S_ID"], 1, 6).alias("CASE_SOURCE"),
    substring(dim_subproduct["SPRD_S_ID"], 8, 2).alias("MENU_CODE"),
    dim_cases_static["CAEX_CUSTOM_TEXT_27"].alias("SOURCE_CODE"),
    dim_subproduct_hier1["SPRDH1_S_ID"].alias("PRODUCT_CODE"),
    dim_subproduct_hier1["SPRDH1_DESC"].alias("PRODUCT_DESC"),
    dim_subproduct["SPRD_S_ID"].alias("SUBPRODUCT_CODE"),
    dim_subproduct["SPRD_DESC"].alias("SUBPRODUCT_DESC"),
    fs_cases_snap["CASE_CREDIT_LIM"].cast(DecimalType(19, 2)).alias("CREDIT_LIMIT"),
    col("LPS2_DESC").alias("CUS_LPS"),
    col("LPS3_DESC").alias("DEBT_LPS"),
    date_format(fs_cases_snap["CASE_LEGAL_PROCESS_STATUS_UPD"], "yyyy-MM-dd").alias("DEBT_LPS_DATE"),
    dim_cases_static["CASE_SOLD_FLAG"].alias("SECURITIZED_FLAG"),
    fs_cases_snap["SPET_CODE"].alias("SPECIAL_TERATMENT"),
    dim_customer_category["CUCAT_DESC"].alias("LEGAL_ENTITY"),
    dim_cases_static["CASE_BRANCH_INTCODE"].alias("BRANCH_CODE"),
    dim_branch["BRN_DESC"].alias("BRANCH_DESC"),
    date_format(fs_cases_snap["CAEX_CUSTOM_DATETIME_12"], "yyyy-MM-dd").alias("COURT_DATE"),
    date_format(fs_cases_snap["CAEX_CUSTOM_DATETIME_11"], "yyyy-MM-dd").alias("PRE_COURT_DATE"),
    when(date_format(dim_cases_static["CSTM_DATETIME_AD_03"], "yyyy-MM-dd") == "9999-12-31", lit("1753-01-01"))
        .otherwise(coalesce(date_format(dim_cases_static["CSTM_DATETIME_AD_03"], "yyyy-MM-dd"), lit("1753-01-01"))).alias("CASE_CLOSE_DATE"),
    coalesce(fs_cases_snap["CAEX_CUSTOM_TEXT_15"], lit("")).alias("SERVICING_TYPE"),
    coalesce(col("PREV_EXTC_NAME"), col("A_PREV_EXTC_NAME")).alias("PREVIOUS_EXTERNAL_AGENCY"),
    coalesce(dim_cases_static["CSTM_TEXT_AD_20"], lit("")).alias("COLLECTOR_ID"),
    coalesce(col("PREV_PACKH3_DESC"), col("A_PREV_PACKH3_DESC")).alias("PREVIOUS_PACKET_PROCESS_TYPE"),
    when(fs_cases_snap_extra["CEXT_GENERIC_BIT_02"] == True, lit("True"))
        .when(fs_cases_snap_extra["CEXT_GENERIC_BIT_02"] == False, lit("False"))
        .otherwise(lit("False")).alias("GUARANTEE_INDICATOR"),
    coalesce(fs_cases_snap_extra["CEXT_GENERIC_NVARCHAR_01"], lit("")).alias("GUARANTEE_ORGANIZATION"),
    when(
        fs_cases_snap_extra["CEXT_GENERIC_NVARCHAR_01"].isNull() |
        fs_cases_snap_extra["CEXT_GENERIC_NVARCHAR_01"].isin("NULL", "", "0"),
        lit("")
    ).otherwise(coalesce(col("LVAL_COMMENTS"), lit(""))).alias("GUARANTEE_ORGANIZATION_DESC"),
    fs_cases_snap["CAEX_CUSTOM_TEXT_44"].alias("SUBSIDY_INDEX"),
    fs_customers_snap["CUEX_GENERIC_BIT_03"].cast(StringType()).alias("CONNECTED_BORROWERS"),
    dim_cases_static["CAEX_CUSTOM_TEXT_16"].alias("LOAN_DURATION"),
    (fs_cases_snap["CSTM_DECIMAL_06"] * fs_cases_snap["MULTIPLY_FOR_LCY"]).cast(DecimalType(19, 2)).alias("ON_BALANCE"),
    (fs_cases_snap["CSTM_DECIMAL_07"] * fs_cases_snap["MULTIPLY_FOR_LCY"]).cast(DecimalType(19, 2)).alias("OFF_BALANCE"),
    (fs_cases_snap["CSTM_DECIMAL_04"] * fs_cases_snap["MULTIPLY_FOR_LCY"]).cast(DecimalType(19, 2)).alias("TOTAL_BALANCE"),
    (fs_cases_snap["CSTM_DECIMAL_05"] * fs_cases_snap["MULTIPLY_FOR_LCY"]).cast(DecimalType(19, 2)).alias("TOTAL_BALANCE_EMLT"),
    dim_settlement_type["SETY_DESC"].alias("SETTLEMENT_TYPE"),
    dim_settlement_status["SEST_DESC"].alias("SETTLEMENT_CURRENT_STATUS"),
    col("L2_DESC").alias("QCCustomerL3869Status"),
    col("L1_DESC").alias("InterbankStatusL3869"),
    fs_customers_snap["CUEX_GENERIC_TEXT_03"].alias("Customer_Watchlist"),
    col("L3_COMMENTS").alias("CustomerL3869"),
    fsad_customers_generic_open["CUSG_GENERIC_DATETIME_07"].cast(StringType()).alias("CourtDate"),
    fsad_customers_generic_open["CUSG_GENERIC_DATETIME_05"].cast(StringType()).alias("ApplicationSubmissionDate"),
    current_timestamp().alias("InsertTimeStamp"),
).distinct()

row_count = final_df.count()
print(f"Final result: {row_count:,} rows | {len(final_df.columns)} columns")

In [0]:
from pyspark.sql.functions import current_timestamp, from_utc_timestamp, lit

target_table = f"{target_area}.cl_fullportfoliorest"

if spark.catalog.tableExists(target_table):
    spark.sql(f"DELETE FROM {target_table} WHERE BusinessDate = '{run_date}'")

final_df_write = final_df.withColumn(
    "IngestionTimestamp",
    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
)

final_df_write.write \
    .format("delta") \
    .partitionBy(*partition_columns) \
    .option("replaceWhere", f"BusinessDate = '{run_date}'") \
    .mode("overwrite") \
    .saveAsTable(target_table)

spark.sql(f"REPAIR TABLE {target_table} SYNC METADATA")

print(f"Written to {target_table} | BusinessDate={run_date}")